# Phase 1B — Prithvi-EO-2.0-300M Linear: Aggregate-Label Benchmark
**Runtime:** Google Colab — **GPU (T4 minimum, A100 preferred)**

**Purpose:** Aggregate-label variant of Prithvi-300M with a linear regression head.  
Strategy: `linear` — entire encoder frozen, only the regression head is trained.
- Encoder: `prithvi_eo_v2_300` (frozen) via TerraTorch `BACKBONE_REGISTRY`
- Head: `LayerNorm(1024) → Dropout(0.1) → Linear(1024, 1)` (scalar output)
- Loss: MSE

Two models are trained sequentially — one per aggregate target:

| Target | Formula | Interpretation |
|---|---|---|
| `coverage_fraction` | n_ookla_tiles_in_chip / total_possible_zoom16_tiles | Spatial coverage |
| `log_mean_tests` | log(1 + total_tests / n_subtiles) | Test density per sub-tile |

Predictions are binarised with a val-optimal threshold and evaluated against the binary
ground-truth labels on the Northeast India geographic hold-out test set.

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + binary labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data (from Google Drive if needed)

## Step 0: Environment Setup

In [ ]:
#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

#
 
C
e
l
l
 
0
.
1
:
 
I
n
s
t
a
l
l
 
d
e
p
e
n
d
e
n
c
i
e
s

#
 
T
e
r
r
a
T
o
r
c
h
 
M
U
S
T
 
b
e
 
i
n
s
t
a
l
l
e
d
 
f
i
r
s
t
 
—
 
r
e
s
t
a
r
t
 
r
u
n
t
i
m
e
 
a
f
t
e
r
 
t
h
i
s
 
c
e
l
l
.

#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

#
 
P
i
n
 
n
u
m
p
y
 
B
E
F
O
R
E
 
t
e
r
r
a
t
o
r
c
h
 
s
o
 
i
t
 
c
a
n
n
o
t
 
b
e
 
u
p
g
r
a
d
e
d
 
t
o
 
2
.
x

#
 
(
p
r
e
v
e
n
t
s
 
"
n
u
m
p
y
.
d
t
y
p
e
 
s
i
z
e
 
c
h
a
n
g
e
d
"
 
A
B
I
 
m
i
s
m
a
t
c
h
 
w
i
t
h
 
p
a
n
d
a
s
/
s
c
i
p
y
)

!
p
i
p
 
i
n
s
t
a
l
l
 
-
q
 
"
n
u
m
p
y
=
=
1
.
2
6
.
4
"

!
p
i
p
 
i
n
s
t
a
l
l
 
-
q
 
t
e
r
r
a
t
o
r
c
h

!
p
i
p
 
i
n
s
t
a
l
l
 
-
q
 
r
a
s
t
e
r
i
o
 
s
c
i
k
i
t
-
l
e
a
r
n
 
g
e
o
p
a
n
d
a
s
 
m
a
t
p
l
o
t
l
i
b
 
s
e
a
b
o
r
n
 
s
c
i
p
y
 
p
y
a
r
r
o
w
 
j
o
b
l
i
b


#
 
R
e
s
t
a
r
t
 
r
u
n
t
i
m
e
 
s
o
 
a
l
l
 
C
 
e
x
t
e
n
s
i
o
n
s
 
l
o
a
d
 
a
g
a
i
n
s
t
 
t
h
e
 
p
i
n
n
e
d
 
n
u
m
p
y

i
m
p
o
r
t
 
o
s

p
r
i
n
t
(
"
R
e
s
t
a
r
t
i
n
g
 
r
u
n
t
i
m
e
 
t
o
 
a
p
p
l
y
 
n
u
m
p
y
 
p
i
n
 
—
 
r
e
-
r
u
n
 
f
r
o
m
 
S
t
e
p
 
0
.
2
 
a
f
t
e
r
 
r
e
s
t
a
r
t
.
"
)

o
s
.
k
i
l
l
(
o
s
.
g
e
t
p
i
d
(
)
,
 
9
)



In [ ]:
# ============================================================
# Cell 0.2: Clone repo
# ============================================================
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# ============================================================
# Cell 0.3: Sync patches from Google Drive
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

## Step 1: Imports & Constants

In [ ]:
i
m
p
o
r
t
 
r
a
n
d
o
m
,
 
n
u
m
p
y
 
a
s
 
n
p
,
 
t
o
r
c
h

r
a
n
d
o
m
.
s
e
e
d
(
4
2
)
;
 
n
p
.
r
a
n
d
o
m
.
s
e
e
d
(
4
2
)

t
o
r
c
h
.
m
a
n
u
a
l
_
s
e
e
d
(
4
2
)
;
 
t
o
r
c
h
.
c
u
d
a
.
m
a
n
u
a
l
_
s
e
e
d
_
a
l
l
(
4
2
)

t
o
r
c
h
.
b
a
c
k
e
n
d
s
.
c
u
d
n
n
.
d
e
t
e
r
m
i
n
i
s
t
i
c
 
=
 
T
r
u
e
;
 
t
o
r
c
h
.
u
s
e
_
d
e
t
e
r
m
i
n
i
s
t
i
c
_
a
l
g
o
r
i
t
h
m
s
(
T
r
u
e
)


i
m
p
o
r
t
 
p
a
n
d
a
s
 
a
s
 
p
d

i
m
p
o
r
t
 
g
e
o
p
a
n
d
a
s
 
a
s
 
g
p
d

i
m
p
o
r
t
 
r
a
s
t
e
r
i
o

i
m
p
o
r
t
 
t
o
r
c
h
.
n
n
 
a
s
 
n
n

i
m
p
o
r
t
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
 
a
s
 
p
l

i
m
p
o
r
t
 
m
a
t
p
l
o
t
l
i
b
.
p
y
p
l
o
t
 
a
s
 
p
l
t

i
m
p
o
r
t
 
w
a
r
n
i
n
g
s

i
m
p
o
r
t
 
l
o
g
g
i
n
g

i
m
p
o
r
t
 
j
s
o
n

f
r
o
m
 
p
a
t
h
l
i
b
 
i
m
p
o
r
t
 
P
a
t
h

f
r
o
m
 
s
h
a
p
e
l
y
.
g
e
o
m
e
t
r
y
 
i
m
p
o
r
t
 
b
o
x

f
r
o
m
 
t
o
r
c
h
.
u
t
i
l
s
.
d
a
t
a
 
i
m
p
o
r
t
 
D
a
t
a
s
e
t
,
 
D
a
t
a
L
o
a
d
e
r

f
r
o
m
 
s
k
l
e
a
r
n
.
m
e
t
r
i
c
s
 
i
m
p
o
r
t
 
(

 
 
 
 
a
v
e
r
a
g
e
_
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
,
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
,

 
 
 
 
r
o
c
_
a
u
c
_
s
c
o
r
e
,
 
f
1
_
s
c
o
r
e
,
 
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
,

 
 
 
 
r
e
c
a
l
l
_
s
c
o
r
e
,
 
a
c
c
u
r
a
c
y
_
s
c
o
r
e
,
 
c
o
n
f
u
s
i
o
n
_
m
a
t
r
i
x
,

 
 
 
 
c
l
a
s
s
i
f
i
c
a
t
i
o
n
_
r
e
p
o
r
t
,
 
m
e
a
n
_
a
b
s
o
l
u
t
e
_
e
r
r
o
r
,
 
m
e
a
n
_
s
q
u
a
r
e
d
_
e
r
r
o
r
,

)

f
r
o
m
 
s
k
l
e
a
r
n
.
m
o
d
e
l
_
s
e
l
e
c
t
i
o
n
 
i
m
p
o
r
t
 
t
r
a
i
n
_
t
e
s
t
_
s
p
l
i
t

f
r
o
m
 
s
c
i
p
y
.
s
t
a
t
s
 
i
m
p
o
r
t
 
s
p
e
a
r
m
a
n
r

f
r
o
m
 
t
q
d
m
.
a
u
t
o
 
i
m
p
o
r
t
 
t
q
d
m


w
a
r
n
i
n
g
s
.
f
i
l
t
e
r
w
a
r
n
i
n
g
s
(
'
i
g
n
o
r
e
'
)

l
o
g
g
i
n
g
.
g
e
t
L
o
g
g
e
r
(
'
r
a
s
t
e
r
i
o
'
)
.
s
e
t
L
e
v
e
l
(
l
o
g
g
i
n
g
.
E
R
R
O
R
)


H
L
S
_
M
E
A
N
S
 
=
 
[
0
.
1
4
2
4
5
4
9
5
,
 
0
.
1
3
9
2
1
4
8
1
,
 
0
.
1
2
4
3
4
6
3
1
,
 
0
.
3
1
4
2
0
0
8
9
,
 
0
.
2
0
7
4
3
5
2
6
,
 
0
.
1
2
0
4
6
5
0
3
]

H
L
S
_
S
T
D
S
 
 
=
 
[
0
.
0
4
0
3
6
2
3
1
,
 
0
.
0
4
1
8
6
9
8
3
,
 
0
.
0
5
2
6
7
6
4
6
,
 
0
.
0
8
2
2
2
2
1
0
,
 
0
.
0
6
8
3
4
7
7
4
,
 
0
.
0
5
2
9
4
2
0
5
]

S
C
A
L
E
 
 
 
 
 
=
 
0
.
0
0
0
1


P
A
T
C
H
_
D
I
R
 
 
 
=
 
'
d
a
t
a
/
r
a
w
/
p
a
t
c
h
e
s
_
2
0
1
9
'

R
E
S
U
L
T
S
_
D
I
R
 
=
 
'
o
u
t
p
u
t
s
/
r
e
s
u
l
t
s
'

P
a
t
h
(
R
E
S
U
L
T
S
_
D
I
R
)
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

P
a
t
h
(
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
'
)
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

P
a
t
h
(
'
o
u
t
p
u
t
s
/
m
o
d
e
l
s
'
)
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

P
a
t
h
(
'
c
h
e
c
k
p
o
i
n
t
s
'
)
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)


#
 
H
L
S
 
c
h
i
p
:
 
2
2
4
 
p
x
 
×
 
3
0
 
m
 
=
 
6
7
2
0
 
m
 
p
e
r
 
s
i
d
e

P
A
T
C
H
_
S
I
Z
E
_
M
 
 
 
=
 
6
7
2
0
.
0

P
A
T
C
H
_
S
I
Z
E
_
D
E
G
 
=
 
P
A
T
C
H
_
S
I
Z
E
_
M
 
/
 
1
1
1
_
3
2
0
.
0

E
A
R
T
H
_
C
I
R
C
_
M
 
 
 
=
 
4
0
_
0
7
5
_
0
1
6
.
6
8
6

Z
O
O
M
1
6
_
S
I
D
E
_
E
Q
 
=
 
E
A
R
T
H
_
C
I
R
C
_
M
 
/
 
(
2
 
*
*
 
1
6
)


D
E
V
I
C
E
 
=
 
'
c
u
d
a
'
 
i
f
 
t
o
r
c
h
.
c
u
d
a
.
i
s
_
a
v
a
i
l
a
b
l
e
(
)
 
e
l
s
e
 
'
c
p
u
'

p
r
i
n
t
(
f
'
U
s
i
n
g
 
d
e
v
i
c
e
:
 
{
D
E
V
I
C
E
}
'
)

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels,
aggregate targets (`coverage_fraction`, `log_mean_tests`),
stratification features, and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

#
#
 
S
t
e
p
 
5
:
 
L
o
a
d
 
C
a
c
h
e
d
 
P
r
i
t
h
v
i
 
E
m
b
e
d
d
i
n
g
s
 
+
 
D
a
t
a
s
e
t

L
o
a
d
 
f
r
o
z
e
n
 
1
0
2
4
-
d
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
f
r
o
m
 
`
o
u
t
p
u
t
s
/
f
e
a
t
u
r
e
s
/
`
 
(
e
x
t
r
a
c
t
e
d
 
b
y
 
N
B
1
6
)
.

I
f
 
t
h
e
 
c
a
c
h
e
 
i
s
 
m
i
s
s
i
n
g
,
 
e
x
t
r
a
c
t
 
a
n
d
 
s
a
v
e
 
t
h
e
m
 
i
n
l
i
n
e
 
—
 
t
h
i
s
 
o
n
l
y
 
h
a
p
p
e
n
s
 
o
n
c
e
.

S
u
b
s
e
q
u
e
n
t
 
e
p
o
c
h
s
 
t
r
a
i
n
 
o
n
l
y
 
t
h
e
 
l
i
g
h
t
w
e
i
g
h
t
 
r
e
g
r
e
s
s
i
o
n
 
h
e
a
d
 
o
n
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
,

w
h
i
c
h
 
i
s
 
m
u
c
h
 
f
a
s
t
e
r
 
a
n
d
 
e
n
s
u
r
e
s
 
c
o
n
s
i
s
t
e
n
c
y
 
a
c
r
o
s
s
 
n
o
t
e
b
o
o
k
s
.

In [ ]:
O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
f
e
a
t
u
r
e
s
'
)

O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)



d
e
f
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
s
p
l
i
t
_
n
a
m
e
,
 
d
f
)
:

 
 
 
 
"
"
"
L
o
a
d
 
c
a
c
h
e
d
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
,
 
o
r
 
e
x
t
r
a
c
t
 
+
 
c
a
c
h
e
 
t
h
e
m
.
"
"
"

 
 
 
 
p
a
t
h
 
=
 
O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
 
/
 
f
'
p
r
i
t
h
v
i
_
f
r
o
z
e
n
_
e
m
b
e
d
s
_
{
s
p
l
i
t
_
n
a
m
e
}
.
n
p
z
'

 
 
 
 
i
f
 
p
a
t
h
.
e
x
i
s
t
s
(
)
:

 
 
 
 
 
 
 
 
d
a
t
a
 
=
 
n
p
.
l
o
a
d
(
p
a
t
h
)

 
 
 
 
 
 
 
 
X
 
=
 
d
a
t
a
[
'
X
'
]
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
'
 
 
L
o
a
d
e
d
 
c
a
c
h
e
d
 
{
s
p
l
i
t
_
n
a
m
e
}
:
 
{
X
.
s
h
a
p
e
}
'
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
X


 
 
 
 
p
r
i
n
t
(
f
'
 
 
C
a
c
h
e
 
m
i
s
s
 
f
o
r
 
{
s
p
l
i
t
_
n
a
m
e
}
 
—
 
e
x
t
r
a
c
t
i
n
g
 
f
r
o
m
 
s
c
r
a
t
c
h
 
.
.
.
'
)

 
 
 
 
f
r
o
m
 
t
e
r
r
a
t
o
r
c
h
.
r
e
g
i
s
t
r
y
 
i
m
p
o
r
t
 
B
A
C
K
B
O
N
E
_
R
E
G
I
S
T
R
Y


 
 
 
 
e
n
c
o
d
e
r
 
=
 
B
A
C
K
B
O
N
E
_
R
E
G
I
S
T
R
Y
.
b
u
i
l
d
(
'
p
r
i
t
h
v
i
_
e
o
_
v
2
_
3
0
0
'
,
 
p
r
e
t
r
a
i
n
e
d
=
T
r
u
e
)
.
t
o
(
D
E
V
I
C
E
)
.
e
v
a
l
(
)

 
 
 
 
f
o
r
 
p
 
i
n
 
e
n
c
o
d
e
r
.
p
a
r
a
m
e
t
e
r
s
(
)
:

 
 
 
 
 
 
 
 
p
.
r
e
q
u
i
r
e
s
_
g
r
a
d
 
=
 
F
a
l
s
e


 
 
 
 
c
l
a
s
s
 
P
r
i
t
h
v
i
P
a
t
c
h
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:

 
 
 
 
 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
d
f
,
 
p
a
t
c
h
_
d
i
r
,
 
n
_
b
a
n
d
s
=
6
)
:

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
d
f
 
=
 
d
f
.
r
e
s
e
t
_
i
n
d
e
x
(
d
r
o
p
=
T
r
u
e
)

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
p
a
t
c
h
_
d
i
r
 
=
 
p
a
t
c
h
_
d
i
r

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
n
_
b
a
n
d
s
 
=
 
n
_
b
a
n
d
s

 
 
 
 
 
 
 
 
d
e
f
 
_
_
l
e
n
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
e
n
(
s
e
l
f
.
d
f
)

 
 
 
 
 
 
 
 
d
e
f
 
_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,
 
i
d
x
)
:

 
 
 
 
 
 
 
 
 
 
 
 
r
o
w
 
=
 
s
e
l
f
.
d
f
.
i
l
o
c
[
i
d
x
]

 
 
 
 
 
 
 
 
 
 
 
 
w
i
t
h
 
r
a
s
t
e
r
i
o
.
o
p
e
n
(
f
"
{
s
e
l
f
.
p
a
t
c
h
_
d
i
r
}
/
{
r
o
w
[
'
p
a
t
c
h
_
i
d
'
]
}
.
t
i
f
"
)
 
a
s
 
s
r
c
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
=
 
s
r
c
.
r
e
a
d
(
l
i
s
t
(
r
a
n
g
e
(
1
,
 
s
e
l
f
.
n
_
b
a
n
d
s
 
+
 
1
)
)
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
*
=
 
S
C
A
L
E

 
 
 
 
 
 
 
 
 
 
 
 
f
o
r
 
b
 
i
n
 
r
a
n
g
e
(
s
e
l
f
.
n
_
b
a
n
d
s
)
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
[
b
]
 
=
 
(
i
m
g
[
b
]
 
-
 
H
L
S
_
M
E
A
N
S
[
b
]
)
 
/
 
H
L
S
_
S
T
D
S
[
b
]

 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
=
 
n
p
.
n
a
n
_
t
o
_
n
u
m
(
i
m
g
,
 
n
a
n
=
0
.
0
)

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
t
o
r
c
h
.
f
r
o
m
_
n
u
m
p
y
(
i
m
g
)
.
u
n
s
q
u
e
e
z
e
(
1
)
,
 
r
o
w
[
'
p
a
t
c
h
_
i
d
'
]


 
 
 
 
d
s
 
=
 
P
r
i
t
h
v
i
P
a
t
c
h
D
a
t
a
s
e
t
(
d
f
,
 
P
A
T
C
H
_
D
I
R
)

 
 
 
 
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(
d
s
,
 
b
a
t
c
h
_
s
i
z
e
=
6
4
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
2
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
)


 
 
 
 
a
l
l
_
e
m
b
s
 
=
 
[
]

 
 
 
 
w
i
t
h
 
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
:

 
 
 
 
 
 
 
 
f
o
r
 
x
,
 
_
 
i
n
 
t
q
d
m
(
l
o
a
d
e
r
,
 
d
e
s
c
=
f
'
E
x
t
r
a
c
t
i
n
g
 
{
s
p
l
i
t
_
n
a
m
e
}
'
)
:

 
 
 
 
 
 
 
 
 
 
 
 
x
 
=
 
x
.
t
o
(
D
E
V
I
C
E
,
 
d
t
y
p
e
=
t
o
r
c
h
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
 
 
 
 
f
e
a
t
s
 
=
 
e
n
c
o
d
e
r
(
x
)

 
 
 
 
 
 
 
 
 
 
 
 
p
o
o
l
e
d
 
=
 
f
e
a
t
s
[
-
1
]
.
m
e
a
n
(
d
i
m
=
1
)

 
 
 
 
 
 
 
 
 
 
 
 
a
l
l
_
e
m
b
s
.
a
p
p
e
n
d
(
p
o
o
l
e
d
.
c
p
u
(
)
.
n
u
m
p
y
(
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)
)


 
 
 
 
X
 
=
 
n
p
.
c
o
n
c
a
t
e
n
a
t
e
(
a
l
l
_
e
m
b
s
,
 
a
x
i
s
=
0
)

 
 
 
 
n
p
.
s
a
v
e
z
_
c
o
m
p
r
e
s
s
e
d
(
p
a
t
h
,
 
X
=
X
,
 
p
a
t
c
h
_
i
d
=
d
f
[
'
p
a
t
c
h
_
i
d
'
]
.
t
o
_
n
u
m
p
y
(
)
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
E
x
t
r
a
c
t
e
d
 
a
n
d
 
c
a
c
h
e
d
 
{
s
p
l
i
t
_
n
a
m
e
}
:
 
{
X
.
s
h
a
p
e
}
'
)


 
 
 
 
d
e
l
 
e
n
c
o
d
e
r

 
 
 
 
t
o
r
c
h
.
c
u
d
a
.
e
m
p
t
y
_
c
a
c
h
e
(
)

 
 
 
 
r
e
t
u
r
n
 
X



p
r
i
n
t
(
'
L
o
a
d
i
n
g
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
(
1
0
2
4
-
d
)
 
.
.
.
'
)

X
_
t
r
a
i
n
_
e
m
b
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
t
r
a
i
n
'
,
 
t
r
a
i
n
_
d
f
)

X
_
v
a
l
_
e
m
b
 
 
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
v
a
l
'
,
 
 
 
v
a
l
_
d
f
)

X
_
t
e
s
t
_
e
m
b
 
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
t
e
s
t
'
,
 
 
t
e
s
t
_
d
f
)

p
r
i
n
t
(
f
'
\
n
T
r
a
i
n
:
 
{
X
_
t
r
a
i
n
_
e
m
b
.
s
h
a
p
e
}
 
 
|
 
 
V
a
l
:
 
{
X
_
v
a
l
_
e
m
b
.
s
h
a
p
e
}
 
 
|
 
 
T
e
s
t
:
 
{
X
_
t
e
s
t
_
e
m
b
.
s
h
a
p
e
}
'
)



c
l
a
s
s
 
E
m
b
e
d
d
i
n
g
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:

 
 
 
 
"
"
"
S
i
m
p
l
e
 
d
a
t
a
s
e
t
 
w
r
a
p
p
i
n
g
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
 
+
 
s
c
a
l
a
r
 
t
a
r
g
e
t
s
.
"
"
"

 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
e
m
b
e
d
d
i
n
g
s
,
 
t
a
r
g
e
t
s
)
:

 
 
 
 
 
 
 
 
s
e
l
f
.
X
 
=
 
t
o
r
c
h
.
f
r
o
m
_
n
u
m
p
y
(
e
m
b
e
d
d
i
n
g
s
)

 
 
 
 
 
 
 
 
s
e
l
f
.
y
 
=
 
t
o
r
c
h
.
t
e
n
s
o
r
(
t
a
r
g
e
t
s
,
 
d
t
y
p
e
=
t
o
r
c
h
.
f
l
o
a
t
3
2
)

 
 
 
 
d
e
f
 
_
_
l
e
n
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
e
n
(
s
e
l
f
.
X
)

 
 
 
 
d
e
f
 
_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,
 
i
d
x
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
s
e
l
f
.
X
[
i
d
x
]
,
 
s
e
l
f
.
y
[
i
d
x
]



_
d
s
 
=
 
E
m
b
e
d
d
i
n
g
D
a
t
a
s
e
t
(
X
_
t
r
a
i
n
_
e
m
b
,
 
t
r
a
i
n
_
d
f
[
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
]
.
v
a
l
u
e
s
)

_
x
,
 
_
y
 
=
 
_
d
s
[
0
]

p
r
i
n
t
(
f
'
E
m
b
e
d
d
i
n
g
 
s
h
a
p
e
:
 
{
_
x
.
s
h
a
p
e
}
 
 
|
 
 
T
a
r
g
e
t
:
 
{
_
y
.
i
t
e
m
(
)
:
.
4
f
}
'
)

d
e
l
 
_
d
s
,
 
_
x
,
 
_
y

#
#
 
S
t
e
p
 
6
:
 
P
r
i
t
h
v
i
R
e
g
r
e
s
s
o
r
 
—
 
H
e
a
d
-
O
n
l
y
 
(
E
m
b
e
d
d
i
n
g
s
 
P
r
e
-
E
x
t
r
a
c
t
e
d
)

S
i
n
c
e
 
e
m
b
e
d
d
i
n
g
s
 
a
r
e
 
p
r
e
-
c
o
m
p
u
t
e
d
 
a
n
d
 
c
a
c
h
e
d
,
 
t
h
e
 
m
o
d
e
l
 
o
n
l
y
 
c
o
n
t
a
i
n
s
 
t
h
e
 
r
e
g
r
e
s
s
i
o
n
 
h
e
a
d
.

N
o
 
e
n
c
o
d
e
r
 
o
n
 
G
P
U
 
d
u
r
i
n
g
 
t
r
a
i
n
i
n
g
 
→
 
f
a
s
t
e
r
 
t
r
a
i
n
i
n
g
 
a
n
d
 
l
o
w
e
r
 
m
e
m
o
r
y
 
u
s
a
g
e
.

In [ ]:
c
l
a
s
s
 
P
r
i
t
h
v
i
R
e
g
r
e
s
s
o
r
(
p
l
.
L
i
g
h
t
n
i
n
g
M
o
d
u
l
e
)
:

 
 
 
 
"
"
"

 
 
 
 
R
e
g
r
e
s
s
i
o
n
 
h
e
a
d
 
t
r
a
i
n
e
d
 
o
n
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
(
1
0
2
4
-
d
)
.

 
 
 
 
H
e
a
d
:
 
L
a
y
e
r
N
o
r
m
(
1
0
2
4
)
 
→
 
D
r
o
p
o
u
t
(
p
)
 
→
 
L
i
n
e
a
r
(
1
0
2
4
,
 
1
)

 
 
 
 
L
o
s
s
:
 
M
S
E

 
 
 
 
"
"
"


 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
t
a
r
g
e
t
_
c
o
l
,
 
l
r
=
1
e
-
3
,
 
w
e
i
g
h
t
_
d
e
c
a
y
=
1
e
-
3
,
 
d
r
o
p
o
u
t
=
0
.
1
)
:

 
 
 
 
 
 
 
 
s
u
p
e
r
(
)
.
_
_
i
n
i
t
_
_
(
)

 
 
 
 
 
 
 
 
s
e
l
f
.
s
a
v
e
_
h
y
p
e
r
p
a
r
a
m
e
t
e
r
s
(
)


 
 
 
 
 
 
 
 
s
e
l
f
.
h
e
a
d
 
=
 
n
n
.
S
e
q
u
e
n
t
i
a
l
(

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
L
a
y
e
r
N
o
r
m
(
1
0
2
4
)
,

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
D
r
o
p
o
u
t
(
d
r
o
p
o
u
t
)
,

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
L
i
n
e
a
r
(
1
0
2
4
,
 
1
)
,

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
s
s
_
f
n
 
=
 
n
n
.
M
S
E
L
o
s
s
(
)


 
 
 
 
 
 
 
 
t
r
a
i
n
a
b
l
e
 
=
 
s
u
m
(
p
.
n
u
m
e
l
(
)
 
f
o
r
 
p
 
i
n
 
s
e
l
f
.
p
a
r
a
m
e
t
e
r
s
(
)
 
i
f
 
p
.
r
e
q
u
i
r
e
s
_
g
r
a
d
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
'
T
a
r
g
e
t
:
 
{
t
a
r
g
e
t
_
c
o
l
}
 
 
|
 
 
T
r
a
i
n
a
b
l
e
 
p
a
r
a
m
s
:
 
{
t
r
a
i
n
a
b
l
e
:
,
}
'
)


 
 
 
 
d
e
f
 
f
o
r
w
a
r
d
(
s
e
l
f
,
 
x
)
:

 
 
 
 
 
 
 
 
"
"
"
x
:
 
(
B
,
 
1
0
2
4
)
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
 
→
 
s
c
a
l
a
r
 
p
r
e
d
i
c
t
i
o
n
s
 
(
B
,
)
"
"
"

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
s
e
l
f
.
h
e
a
d
(
x
)
.
s
q
u
e
e
z
e
(
1
)


 
 
 
 
d
e
f
 
t
r
a
i
n
i
n
g
_
s
t
e
p
(
s
e
l
f
,
 
b
a
t
c
h
,
 
b
a
t
c
h
_
i
d
x
)
:

 
 
 
 
 
 
 
 
x
,
 
y
 
 
 
=
 
b
a
t
c
h

 
 
 
 
 
 
 
 
p
r
e
d
s
 
 
=
 
s
e
l
f
(
x
)

 
 
 
 
 
 
 
 
l
o
s
s
 
 
 
=
 
s
e
l
f
.
l
o
s
s
_
f
n
(
p
r
e
d
s
,
 
y
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
t
r
a
i
n
_
l
o
s
s
'
,
 
l
o
s
s
,
 
p
r
o
g
_
b
a
r
=
T
r
u
e
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
o
s
s


 
 
 
 
d
e
f
 
v
a
l
i
d
a
t
i
o
n
_
s
t
e
p
(
s
e
l
f
,
 
b
a
t
c
h
,
 
b
a
t
c
h
_
i
d
x
)
:

 
 
 
 
 
 
 
 
x
,
 
y
 
 
=
 
b
a
t
c
h

 
 
 
 
 
 
 
 
p
r
e
d
s
 
=
 
s
e
l
f
(
x
)

 
 
 
 
 
 
 
 
l
o
s
s
 
 
=
 
s
e
l
f
.
l
o
s
s
_
f
n
(
p
r
e
d
s
,
 
y
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
v
a
l
_
l
o
s
s
'
,
 
l
o
s
s
,
 
p
r
o
g
_
b
a
r
=
T
r
u
e
,
 
s
y
n
c
_
d
i
s
t
=
T
r
u
e
)


 
 
 
 
d
e
f
 
c
o
n
f
i
g
u
r
e
_
o
p
t
i
m
i
z
e
r
s
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
o
p
t
i
m
i
z
e
r
 
=
 
t
o
r
c
h
.
o
p
t
i
m
.
A
d
a
m
W
(

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
p
a
r
a
m
e
t
e
r
s
(
)
,

 
 
 
 
 
 
 
 
 
 
 
 
l
r
=
s
e
l
f
.
h
p
a
r
a
m
s
.
l
r
,

 
 
 
 
 
 
 
 
 
 
 
 
w
e
i
g
h
t
_
d
e
c
a
y
=
s
e
l
f
.
h
p
a
r
a
m
s
.
w
e
i
g
h
t
_
d
e
c
a
y
,

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
s
c
h
e
d
u
l
e
r
 
=
 
t
o
r
c
h
.
o
p
t
i
m
.
l
r
_
s
c
h
e
d
u
l
e
r
.
R
e
d
u
c
e
L
R
O
n
P
l
a
t
e
a
u
(

 
 
 
 
 
 
 
 
 
 
 
 
o
p
t
i
m
i
z
e
r
,
 
m
o
d
e
=
'
m
i
n
'
,
 
f
a
c
t
o
r
=
0
.
5
,
 
p
a
t
i
e
n
c
e
=
1
0

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
{

 
 
 
 
 
 
 
 
 
 
 
 
'
o
p
t
i
m
i
z
e
r
'
:
 
o
p
t
i
m
i
z
e
r
,

 
 
 
 
 
 
 
 
 
 
 
 
'
l
r
_
s
c
h
e
d
u
l
e
r
'
:
 
{
'
s
c
h
e
d
u
l
e
r
'
:
 
s
c
h
e
d
u
l
e
r
,
 
'
m
o
n
i
t
o
r
'
:
 
'
v
a
l
_
l
o
s
s
'
}
,

 
 
 
 
 
 
 
 
}

#
#
 
S
t
e
p
 
7
:
 
T
r
a
i
n
i
n
g
 
—
 
B
o
t
h
 
T
a
r
g
e
t
s
 
(
F
i
x
e
d
 
C
o
n
s
e
r
v
a
t
i
v
e
 
D
e
f
a
u
l
t
s
)


D
u
e
 
t
o
 
p
r
o
j
e
c
t
 
t
i
m
e
 
c
o
n
s
t
r
a
i
n
t
s
,
 
t
h
e
 
l
i
n
e
a
r
 
h
e
a
d
 
i
s
 
t
r
a
i
n
e
d
 
w
i
t
h
 
a
 
s
m
a
l
l
 
s
e
t
 
o
f

c
o
n
s
e
r
v
a
t
i
v
e
 
d
e
f
a
u
l
t
 
h
y
p
e
r
p
a
r
a
m
e
t
e
r
s
 
c
h
o
s
e
n
 
t
o
 
p
r
i
o
r
i
t
i
z
e
 
s
t
a
b
i
l
i
t
y
 
u
n
d
e
r

g
e
o
g
r
a
p
h
i
c
 
d
i
s
t
r
i
b
u
t
i
o
n
 
s
h
i
f
t
.


|
 
P
a
r
a
m
 
|
 
V
a
l
u
e
 
|

|
-
-
-
|
-
-
-
|

|
 
`
l
e
a
r
n
i
n
g
_
r
a
t
e
`
 
|
 
1
e
-
3
 
|

|
 
`
w
e
i
g
h
t
_
d
e
c
a
y
`
 
|
 
1
e
-
3
 
|

|
 
`
d
r
o
p
o
u
t
`
 
|
 
0
.
1
 
|

|
 
`
b
a
t
c
h
_
s
i
z
e
`
 
|
 
3
2
 
|

|
 
`
m
a
x
_
e
p
o
c
h
s
`
 
|
 
1
0
0
 
|

|
 
`
e
a
r
l
y
_
s
t
o
p
p
i
n
g
_
p
a
t
i
e
n
c
e
`
 
|
 
1
5
 
|

|
 
`
o
p
t
i
m
i
z
e
r
`
 
|
 
A
d
a
m
W
 
|

|
 
`
l
o
s
s
`
 
|
 
M
S
E
L
o
s
s
 
|

In [ ]:
f
r
o
m
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
 
i
m
p
o
r
t
 
T
r
a
i
n
e
r

f
r
o
m
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
.
c
a
l
l
b
a
c
k
s
 
i
m
p
o
r
t
 
E
a
r
l
y
S
t
o
p
p
i
n
g
,
 
M
o
d
e
l
C
h
e
c
k
p
o
i
n
t



c
l
a
s
s
 
M
e
t
r
i
c
T
r
a
c
k
e
r
(
p
l
.
C
a
l
l
b
a
c
k
)
:

 
 
 
 
"
"
"
R
e
c
o
r
d
s
 
v
a
l
_
l
o
s
s
 
(
a
n
d
 
t
r
a
i
n
_
l
o
s
s
)
 
a
t
 
t
h
e
 
e
n
d
 
o
f
 
e
v
e
r
y
 
e
p
o
c
h
.
"
"
"

 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
s
e
l
f
.
t
r
a
i
n
_
l
o
s
s
 
=
 
[
]

 
 
 
 
 
 
 
 
s
e
l
f
.
v
a
l
_
l
o
s
s
 
 
 
=
 
[
]


 
 
 
 
d
e
f
 
o
n
_
t
r
a
i
n
_
e
p
o
c
h
_
e
n
d
(
s
e
l
f
,
 
t
r
a
i
n
e
r
,
 
p
l
_
m
o
d
u
l
e
)
:

 
 
 
 
 
 
 
 
v
 
=
 
t
r
a
i
n
e
r
.
c
a
l
l
b
a
c
k
_
m
e
t
r
i
c
s
.
g
e
t
(
'
t
r
a
i
n
_
l
o
s
s
'
)

 
 
 
 
 
 
 
 
s
e
l
f
.
t
r
a
i
n
_
l
o
s
s
.
a
p
p
e
n
d
(
f
l
o
a
t
(
v
)
 
i
f
 
v
 
i
s
 
n
o
t
 
N
o
n
e
 
e
l
s
e
 
f
l
o
a
t
(
'
n
a
n
'
)
)


 
 
 
 
d
e
f
 
o
n
_
v
a
l
i
d
a
t
i
o
n
_
e
p
o
c
h
_
e
n
d
(
s
e
l
f
,
 
t
r
a
i
n
e
r
,
 
p
l
_
m
o
d
u
l
e
)
:

 
 
 
 
 
 
 
 
v
 
=
 
t
r
a
i
n
e
r
.
c
a
l
l
b
a
c
k
_
m
e
t
r
i
c
s
.
g
e
t
(
'
v
a
l
_
l
o
s
s
'
)

 
 
 
 
 
 
 
 
s
e
l
f
.
v
a
l
_
l
o
s
s
.
a
p
p
e
n
d
(
f
l
o
a
t
(
v
)
 
i
f
 
v
 
i
s
 
n
o
t
 
N
o
n
e
 
e
l
s
e
 
f
l
o
a
t
(
'
n
a
n
'
)
)



#
 
─
─
 
F
i
x
e
d
 
c
o
n
s
e
r
v
a
t
i
v
e
 
d
e
f
a
u
l
t
s
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

F
I
X
E
D
_
C
F
G
 
=
 
{

 
 
 
 
'
l
r
'
:
 
 
 
 
 
 
 
 
 
 
 
1
e
-
3
,

 
 
 
 
'
w
e
i
g
h
t
_
d
e
c
a
y
'
:
 
1
e
-
3
,

 
 
 
 
'
d
r
o
p
o
u
t
'
:
 
 
 
 
 
 
0
.
1
,

 
 
 
 
'
b
a
t
c
h
_
s
i
z
e
'
:
 
 
 
3
2
,

 
 
 
 
'
m
a
x
_
e
p
o
c
h
s
'
:
 
 
 
1
0
0
,

 
 
 
 
'
p
a
t
i
e
n
c
e
'
:
 
 
 
 
 
1
5
,

}


p
r
i
n
t
(
f
'
F
i
x
e
d
 
c
o
n
f
i
g
:
 
{
F
I
X
E
D
_
C
F
G
}
'
)


#
 
─
─
 
T
r
a
i
n
i
n
g
 
p
e
r
 
t
a
r
g
e
t
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

b
e
s
t
_
c
k
p
t
_
p
a
t
h
s
 
=
 
{
}

v
a
l
_
h
i
s
t
o
r
i
e
s
 
 
 
=
 
{
}


f
o
r
 
T
A
R
G
E
T
_
N
A
M
E
 
i
n
 
[
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
,
 
'
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
]
:

 
 
 
 
p
r
i
n
t
(
f
'
\
n
{
"
=
"
*
7
0
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
T
R
A
I
N
I
N
G
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
{
"
=
"
*
7
0
}
'
)


 
 
 
 
t
r
a
i
n
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
 
 
 
 
E
m
b
e
d
d
i
n
g
D
a
t
a
s
e
t
(
X
_
t
r
a
i
n
_
e
m
b
,
 
t
r
a
i
n
_
d
f
[
T
A
R
G
E
T
_
N
A
M
E
]
.
v
a
l
u
e
s
)
,

 
 
 
 
 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
F
I
X
E
D
_
C
F
G
[
'
b
a
t
c
h
_
s
i
z
e
'
]
,
 
s
h
u
f
f
l
e
=
T
r
u
e
,

 
 
 
 
 
 
 
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
,

 
 
 
 
)

 
 
 
 
v
a
l
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
 
 
 
 
E
m
b
e
d
d
i
n
g
D
a
t
a
s
e
t
(
X
_
v
a
l
_
e
m
b
,
 
v
a
l
_
d
f
[
T
A
R
G
E
T
_
N
A
M
E
]
.
v
a
l
u
e
s
)
,

 
 
 
 
 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
F
I
X
E
D
_
C
F
G
[
'
b
a
t
c
h
_
s
i
z
e
'
]
 
*
 
2
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,

 
 
 
 
 
 
 
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
,

 
 
 
 
)


 
 
 
 
m
o
d
e
l
 
=
 
P
r
i
t
h
v
i
R
e
g
r
e
s
s
o
r
(

 
 
 
 
 
 
 
 
t
a
r
g
e
t
_
c
o
l
=
T
A
R
G
E
T
_
N
A
M
E
,

 
 
 
 
 
 
 
 
l
r
=
F
I
X
E
D
_
C
F
G
[
'
l
r
'
]
,

 
 
 
 
 
 
 
 
w
e
i
g
h
t
_
d
e
c
a
y
=
F
I
X
E
D
_
C
F
G
[
'
w
e
i
g
h
t
_
d
e
c
a
y
'
]
,

 
 
 
 
 
 
 
 
d
r
o
p
o
u
t
=
F
I
X
E
D
_
C
F
G
[
'
d
r
o
p
o
u
t
'
]
,

 
 
 
 
)

 
 
 
 
m
e
t
r
i
c
_
t
r
a
c
k
e
r
 
=
 
M
e
t
r
i
c
T
r
a
c
k
e
r
(
)

 
 
 
 
e
a
r
l
y
_
s
t
o
p
 
=
 
E
a
r
l
y
S
t
o
p
p
i
n
g
(

 
 
 
 
 
 
 
 
m
o
n
i
t
o
r
=
'
v
a
l
_
l
o
s
s
'
,
 
p
a
t
i
e
n
c
e
=
F
I
X
E
D
_
C
F
G
[
'
p
a
t
i
e
n
c
e
'
]
,

 
 
 
 
 
 
 
 
m
o
d
e
=
'
m
i
n
'
,
 
v
e
r
b
o
s
e
=
T
r
u
e
,

 
 
 
 
)

 
 
 
 
c
h
e
c
k
p
o
i
n
t
_
c
b
 
=
 
M
o
d
e
l
C
h
e
c
k
p
o
i
n
t
(

 
 
 
 
 
 
 
 
d
i
r
p
a
t
h
=
'
c
h
e
c
k
p
o
i
n
t
s
'
,

 
 
 
 
 
 
 
 
f
i
l
e
n
a
m
e
=
f
'
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
'
 
+
 
'
{
e
p
o
c
h
:
0
2
d
}
_
{
v
a
l
_
l
o
s
s
:
.
4
f
}
'
,

 
 
 
 
 
 
 
 
m
o
n
i
t
o
r
=
'
v
a
l
_
l
o
s
s
'
,
 
m
o
d
e
=
'
m
i
n
'
,
 
s
a
v
e
_
t
o
p
_
k
=
1
,
 
v
e
r
b
o
s
e
=
T
r
u
e
,

 
 
 
 
)


 
 
 
 
t
r
a
i
n
e
r
 
=
 
T
r
a
i
n
e
r
(

 
 
 
 
 
 
 
 
m
a
x
_
e
p
o
c
h
s
=
F
I
X
E
D
_
C
F
G
[
'
m
a
x
_
e
p
o
c
h
s
'
]
,

 
 
 
 
 
 
 
 
a
c
c
e
l
e
r
a
t
o
r
=
'
g
p
u
'
 
i
f
 
t
o
r
c
h
.
c
u
d
a
.
i
s
_
a
v
a
i
l
a
b
l
e
(
)
 
e
l
s
e
 
'
c
p
u
'
,

 
 
 
 
 
 
 
 
d
e
v
i
c
e
s
=
1
,

 
 
 
 
 
 
 
 
c
a
l
l
b
a
c
k
s
=
[
m
e
t
r
i
c
_
t
r
a
c
k
e
r
,
 
e
a
r
l
y
_
s
t
o
p
,
 
c
h
e
c
k
p
o
i
n
t
_
c
b
]
,

 
 
 
 
 
 
 
 
l
o
g
_
e
v
e
r
y
_
n
_
s
t
e
p
s
=
1
0
,

 
 
 
 
 
 
 
 
e
n
a
b
l
e
_
p
r
o
g
r
e
s
s
_
b
a
r
=
T
r
u
e
,

 
 
 
 
)

 
 
 
 
t
r
a
i
n
e
r
.
f
i
t
(
m
o
d
e
l
,
 
t
r
a
i
n
_
l
o
a
d
e
r
,
 
v
a
l
_
l
o
a
d
e
r
)


 
 
 
 
b
e
s
t
_
c
k
p
t
_
p
a
t
h
s
[
T
A
R
G
E
T
_
N
A
M
E
]
 
=
 
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
p
a
t
h

 
 
 
 
v
a
l
_
h
i
s
t
o
r
i
e
s
[
T
A
R
G
E
T
_
N
A
M
E
]
 
 
 
=
 
m
e
t
r
i
c
_
t
r
a
c
k
e
r
.
v
a
l
_
l
o
s
s

 
 
 
 
p
r
i
n
t
(
f
'
B
e
s
t
 
c
h
e
c
k
p
o
i
n
t
:
 
{
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
p
a
t
h
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
B
e
s
t
 
v
a
l
 
l
o
s
s
 
 
:
 
{
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
s
c
o
r
e
:
.
6
f
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
T
r
a
i
n
e
d
 
e
p
o
c
h
s
 
:
 
{
l
e
n
(
m
e
t
r
i
c
_
t
r
a
c
k
e
r
.
v
a
l
_
l
o
s
s
)
}
'
)


 
 
 
 
d
e
l
 
m
o
d
e
l
,
 
t
r
a
i
n
e
r
,
 
t
r
a
i
n
_
l
o
a
d
e
r
,
 
v
a
l
_
l
o
a
d
e
r

 
 
 
 
t
o
r
c
h
.
c
u
d
a
.
e
m
p
t
y
_
c
a
c
h
e
(
)


p
r
i
n
t
(
'
\
n
 
A
l
l
 
t
r
a
i
n
i
n
g
 
c
o
m
p
l
e
t
e
.
'
)

p
r
i
n
t
(
'
C
h
e
c
k
p
o
i
n
t
s
:
'
,
 
b
e
s
t
_
c
k
p
t
_
p
a
t
h
s
)

#
#
 
S
t
e
p
 
8
:
 
E
v
a
l
u
a
t
i
o
n


P
e
r
 
t
a
r
g
e
t
,
 
f
o
u
r
 
d
i
a
g
n
o
s
t
i
c
 
p
l
o
t
 
g
r
o
u
p
s
 
a
r
e
 
p
r
o
d
u
c
e
d
:


|
 
F
i
g
u
r
e
 
|
 
C
o
n
t
e
n
t
 
|
 
Q
u
e
s
t
i
o
n
 
a
n
s
w
e
r
e
d
 
|

|
-
-
-
|
-
-
-
|
-
-
-
|

|
 
1
 
|
 
V
a
l
i
d
a
t
i
o
n
 
l
o
s
s
 
o
v
e
r
 
e
p
o
c
h
 
|
 
D
i
d
 
t
r
a
i
n
i
n
g
 
c
o
n
v
e
r
g
e
?
 
W
h
e
n
 
d
i
d
 
i
t
 
s
t
o
p
?
 
|

|
 
2
 
|
 
P
r
e
d
i
c
t
e
d
 
v
s
 
a
c
t
u
a
l
 
s
c
a
t
t
e
r
 
+
 
M
A
E
/
R
M
S
E
 
b
a
r
 
|
 
H
o
w
 
w
e
l
l
 
d
o
e
s
 
t
h
e
 
m
o
d
e
l
 
t
r
a
c
k
 
t
h
e
 
c
o
n
t
i
n
u
o
u
s
 
t
a
r
g
e
t
?
 
|

|
 
3
 
|
 
R
e
s
i
d
u
a
l
s
 
v
s
 
t
r
u
e
 
v
a
l
u
e
 
|
 
W
h
e
r
e
 
d
o
e
s
 
t
h
e
 
m
o
d
e
l
 
s
t
r
u
g
g
l
e
 
(
l
o
w
 
v
s
 
h
i
g
h
 
d
e
n
s
i
t
y
 
c
h
i
p
s
)
?
 
|

|
 
4
 
|
 
S
p
a
t
i
a
l
 
p
r
e
d
i
c
t
i
o
n
 
m
a
p
 
(
l
o
n
/
l
a
t
)
 
|
 
D
o
e
s
 
t
h
e
 
m
o
d
e
l
 
l
e
a
r
n
 
c
o
h
e
r
e
n
t
 
s
p
a
t
i
a
l
 
p
a
t
t
e
r
n
s
?
 
|

|
 
5
 
|
 
C
o
n
f
u
s
i
o
n
 
m
a
t
r
i
x
 
+
 
P
R
 
c
u
r
v
e
 
|
 
D
o
e
s
 
t
h
e
 
c
o
n
t
i
n
u
o
u
s
 
s
c
o
r
e
 
h
e
l
p
 
o
n
 
t
h
e
 
b
i
n
a
r
y
 
c
o
n
n
e
c
t
i
v
i
t
y
 
t
a
s
k
?
 
|


S
e
l
f
-
c
o
n
t
a
i
n
e
d
:
 
r
e
s
o
l
v
e
s
 
c
h
e
c
k
p
o
i
n
t
s
 
f
r
o
m
 
`
c
h
e
c
k
p
o
i
n
t
s
/
`
 
d
i
r
e
c
t
o
r
y
.

In [ ]:
# ============================================================
# Cell 8.1: Checkpoint resolution helper
# Priority 1: best_ckpt_paths dict from training above (in-session)
# Priority 2: scan checkpoints/ directory for most recent matching file
# ============================================================
def resolve_checkpoint(target_name):
    # Priority 1: in-session training result
    if 'best_ckpt_paths' in globals() and target_name in best_ckpt_paths:
        p = best_ckpt_paths[target_name]
        if Path(p).exists():
            print(f'[{target_name}] Using in-session checkpoint: {p}')
            return p
    # Priority 2: scan checkpoints/
    candidates = sorted(
        Path('checkpoints').glob(f'prithvi_linear_agg_{target_name}_*.ckpt'),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    if candidates:
        print(f'[{target_name}] Found checkpoint: {candidates[0]}')
        return str(candidates[0])
    raise FileNotFoundError(
        f'No checkpoint found for target "{target_name}". Run training (Step 7) first.'
    )

In [ ]:
#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

#
 
C
e
l
l
 
8
.
2
:
 
I
n
f
e
r
e
n
c
e
 
h
e
l
p
e
r
 
—
 
u
s
e
s
 
c
a
c
h
e
d
 
e
m
b
e
d
d
i
n
g
s

#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

d
e
f
 
r
u
n
_
i
n
f
e
r
e
n
c
e
(
m
o
d
e
l
,
 
e
m
b
e
d
d
i
n
g
s
,
 
t
a
r
g
e
t
s
,
 
b
a
t
c
h
_
s
i
z
e
=
2
5
6
)
:

 
 
 
 
"
"
"
R
u
n
 
h
e
a
d
-
o
n
l
y
 
i
n
f
e
r
e
n
c
e
 
o
n
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
.
"
"
"

 
 
 
 
d
a
t
a
s
e
t
 
=
 
E
m
b
e
d
d
i
n
g
D
a
t
a
s
e
t
(
e
m
b
e
d
d
i
n
g
s
,
 
t
a
r
g
e
t
s
)

 
 
 
 
l
o
a
d
e
r
 
 
=
 
D
a
t
a
L
o
a
d
e
r
(
d
a
t
a
s
e
t
,
 
b
a
t
c
h
_
s
i
z
e
=
b
a
t
c
h
_
s
i
z
e
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
0
)

 
 
 
 
p
r
e
d
s
_
l
i
s
t
 
=
 
[
]

 
 
 
 
m
o
d
e
l
.
e
v
a
l
(
)

 
 
 
 
w
i
t
h
 
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
:

 
 
 
 
 
 
 
 
f
o
r
 
x
,
 
_
 
i
n
 
l
o
a
d
e
r
:

 
 
 
 
 
 
 
 
 
 
 
 
p
r
e
d
s
_
l
i
s
t
.
e
x
t
e
n
d
(
m
o
d
e
l
(
x
.
t
o
(
D
E
V
I
C
E
)
)
.
c
p
u
(
)
.
n
u
m
p
y
(
)
)

 
 
 
 
r
e
t
u
r
n
 
n
p
.
a
r
r
a
y
(
p
r
e
d
s
_
l
i
s
t
)
,
 
t
a
r
g
e
t
s

In [ ]:
#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

#
 
C
e
l
l
 
8
.
3
:
 
E
v
a
l
u
a
t
e
 
b
o
t
h
 
t
a
r
g
e
t
s

#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

f
r
o
m
 
s
c
i
p
y
.
s
t
a
t
s
 
i
m
p
o
r
t
 
b
i
n
n
e
d
_
s
t
a
t
i
s
t
i
c


a
l
l
_
m
e
t
r
i
c
s
 
=
 
{
}


f
o
r
 
T
A
R
G
E
T
_
N
A
M
E
 
i
n
 
[
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
,
 
'
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
]
:

 
 
 
 
p
r
i
n
t
(
f
'
\
n
{
"
=
"
*
6
0
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
E
v
a
l
u
a
t
i
n
g
:
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
{
"
=
"
*
6
0
}
'
)


 
 
 
 
c
k
p
t
_
p
a
t
h
 
=
 
r
e
s
o
l
v
e
_
c
h
e
c
k
p
o
i
n
t
(
T
A
R
G
E
T
_
N
A
M
E
)

 
 
 
 
r
e
g
_
m
o
d
e
l
 
=
 
P
r
i
t
h
v
i
R
e
g
r
e
s
s
o
r
.
l
o
a
d
_
f
r
o
m
_
c
h
e
c
k
p
o
i
n
t
(
c
k
p
t
_
p
a
t
h
)
.
t
o
(
D
E
V
I
C
E
)
.
e
v
a
l
(
)

 
 
 
 
p
r
i
n
t
(
'
M
o
d
e
l
 
l
o
a
d
e
d
.
'
)


 
 
 
 
#
 
─
─
 
F
i
g
u
r
e
 
1
:
 
V
a
l
i
d
a
t
i
o
n
 
l
o
s
s
 
o
v
e
r
 
e
p
o
c
h
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
h
i
s
t
 
=
 
v
a
l
_
h
i
s
t
o
r
i
e
s
.
g
e
t
(
T
A
R
G
E
T
_
N
A
M
E
,
 
[
]
)
 
i
f
 
'
v
a
l
_
h
i
s
t
o
r
i
e
s
'
 
i
n
 
g
l
o
b
a
l
s
(
)
 
e
l
s
e
 
[
]

 
 
 
 
i
f
 
h
i
s
t
:

 
 
 
 
 
 
 
 
f
i
g
,
 
a
x
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
f
i
g
s
i
z
e
=
(
8
,
 
4
)
)

 
 
 
 
 
 
 
 
e
p
o
c
h
s
 
 
 
 
 
=
 
l
i
s
t
(
r
a
n
g
e
(
1
,
 
l
e
n
(
h
i
s
t
)
 
+
 
1
)
)

 
 
 
 
 
 
 
 
b
e
s
t
_
e
p
o
c
h
 
=
 
i
n
t
(
n
p
.
a
r
g
m
i
n
(
h
i
s
t
)
)
 
+
 
1

 
 
 
 
 
 
 
 
a
x
.
p
l
o
t
(
e
p
o
c
h
s
,
 
h
i
s
t
,
 
l
w
=
2
,
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
,
 
l
a
b
e
l
=
'
V
a
l
 
l
o
s
s
 
(
M
S
E
)
'
)

 
 
 
 
 
 
 
 
a
x
.
a
x
v
l
i
n
e
(
b
e
s
t
_
e
p
o
c
h
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
a
l
p
h
a
=
0
.
8
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
B
e
s
t
 
e
p
o
c
h
 
{
b
e
s
t
_
e
p
o
c
h
}
 
 
(
l
o
s
s
 
=
 
{
m
i
n
(
h
i
s
t
)
:
.
4
f
}
)
'
)

 
 
 
 
 
 
 
 
a
x
.
s
e
t
_
x
l
a
b
e
l
(
'
E
p
o
c
h
'
)
;
 
a
x
.
s
e
t
_
y
l
a
b
e
l
(
'
V
a
l
 
M
S
E
 
L
o
s
s
'
)

 
 
 
 
 
 
 
 
a
x
.
s
e
t
_
t
i
t
l
e
(
f
'
V
a
l
i
d
a
t
i
o
n
 
L
o
s
s
 
o
v
e
r
 
E
p
o
c
h
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
 
 
 
 
a
x
.
l
e
g
e
n
d
(
)
;
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
 
 
 
 
p
l
t
.
s
a
v
e
f
i
g
(
f
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
v
a
l
_
l
o
s
s
.
p
n
g
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

 
 
 
 
 
 
 
 
p
l
t
.
s
h
o
w
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
'
(
v
a
l
_
h
i
s
t
o
r
i
e
s
 
n
o
t
 
a
v
a
i
l
a
b
l
e
 
—
 
r
u
n
 
t
r
a
i
n
i
n
g
 
f
i
r
s
t
 
t
o
 
s
e
e
 
l
e
a
r
n
i
n
g
 
c
u
r
v
e
)
'
)


 
 
 
 
#
 
─
─
 
V
a
l
 
i
n
f
e
r
e
n
c
e
 
→
 
o
p
t
i
m
a
l
 
b
i
n
a
r
i
s
a
t
i
o
n
 
t
h
r
e
s
h
o
l
d
 
─
─
─
─
─
─
─

 
 
 
 
v
a
l
_
p
r
e
d
s
,
 
_
 
=
 
r
u
n
_
i
n
f
e
r
e
n
c
e
(
r
e
g
_
m
o
d
e
l
,
 
X
_
v
a
l
_
e
m
b
,
 
v
a
l
_
d
f
[
T
A
R
G
E
T
_
N
A
M
E
]
.
v
a
l
u
e
s
)

 
 
 
 
y
_
v
a
l
_
b
i
n
 
 
 
 
=
 
v
a
l
_
d
f
[
'
h
a
s
_
c
o
v
e
r
a
g
e
'
]
.
v
a
l
u
e
s


 
 
 
 
p
r
e
c
_
v
,
 
r
e
c
_
v
,
 
t
h
r
_
v
 
=
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
(
y
_
v
a
l
_
b
i
n
,
 
v
a
l
_
p
r
e
d
s
)

 
 
 
 
f
1
_
v
 
 
 
 
=
 
2
 
*
 
p
r
e
c
_
v
[
:
-
1
]
 
*
 
r
e
c
_
v
[
:
-
1
]
 
/
 
(
p
r
e
c
_
v
[
:
-
1
]
 
+
 
r
e
c
_
v
[
:
-
1
]
 
+
 
1
e
-
8
)

 
 
 
 
o
p
t
_
t
h
r
 
=
 
f
l
o
a
t
(
t
h
r
_
v
[
n
p
.
a
r
g
m
a
x
(
f
1
_
v
)
]
)

 
 
 
 
p
r
i
n
t
(
f
'
V
a
l
-
o
p
t
i
m
a
l
 
t
h
r
e
s
h
o
l
d
:
 
{
o
p
t
_
t
h
r
:
.
4
f
}
'
)


 
 
 
 
#
 
─
─
 
T
e
s
t
 
i
n
f
e
r
e
n
c
e
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
t
e
s
t
_
p
r
e
d
s
,
 
_
 
=
 
r
u
n
_
i
n
f
e
r
e
n
c
e
(
r
e
g
_
m
o
d
e
l
,
 
X
_
t
e
s
t
_
e
m
b
,
 
t
e
s
t
_
d
f
[
T
A
R
G
E
T
_
N
A
M
E
]
.
v
a
l
u
e
s
)

 
 
 
 
y
_
t
e
s
t
_
b
i
n
 
 
 
 
=
 
t
e
s
t
_
d
f
[
'
h
a
s
_
c
o
v
e
r
a
g
e
'
]
.
v
a
l
u
e
s

 
 
 
 
y
_
t
e
s
t
_
c
o
n
t
 
 
 
=
 
t
e
s
t
_
d
f
[
T
A
R
G
E
T
_
N
A
M
E
]
.
v
a
l
u
e
s

 
 
 
 
y
_
p
r
e
d
_
b
i
n
 
 
 
 
=
 
(
t
e
s
t
_
p
r
e
d
s
 
>
=
 
o
p
t
_
t
h
r
)
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
r
e
s
i
d
u
a
l
s
 
 
 
 
 
=
 
t
e
s
t
_
p
r
e
d
s
 
-
 
y
_
t
e
s
t
_
c
o
n
t


 
 
 
 
#
 
─
─
 
P
R
I
M
A
R
Y
:
 
r
e
g
r
e
s
s
i
o
n
 
q
u
a
l
i
t
y
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
m
a
e
 
 
 
 
=
 
m
e
a
n
_
a
b
s
o
l
u
t
e
_
e
r
r
o
r
(
y
_
t
e
s
t
_
c
o
n
t
,
 
t
e
s
t
_
p
r
e
d
s
)

 
 
 
 
r
m
s
e
 
 
 
=
 
n
p
.
s
q
r
t
(
m
e
a
n
_
s
q
u
a
r
e
d
_
e
r
r
o
r
(
y
_
t
e
s
t
_
c
o
n
t
,
 
t
e
s
t
_
p
r
e
d
s
)
)

 
 
 
 
r
h
o
,
 
_
 
=
 
s
p
e
a
r
m
a
n
r
(
y
_
t
e
s
t
_
c
o
n
t
,
 
t
e
s
t
_
p
r
e
d
s
)


 
 
 
 
p
r
i
n
t
(
'
\
n
-
-
-
 
P
R
I
M
A
R
Y
:
 
R
e
g
r
e
s
s
i
o
n
 
q
u
a
l
i
t
y
 
-
-
-
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
M
A
E
 
 
 
 
 
 
 
 
 
 
:
 
{
m
a
e
:
.
4
f
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
R
M
S
E
 
 
 
 
 
 
 
 
 
:
 
{
r
m
s
e
:
.
4
f
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
S
p
e
a
r
m
a
n
 
r
h
o
 
:
 
{
r
h
o
:
.
4
f
}
'
)


 
 
 
 
#
 
─
─
 
S
E
C
O
N
D
A
R
Y
:
 
b
i
n
a
r
y
 
c
o
n
n
e
c
t
i
v
i
t
y
 
t
a
s
k
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
p
r
_
a
u
c
 
 
 
=
 
a
v
e
r
a
g
e
_
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
t
e
s
t
_
p
r
e
d
s
)

 
 
 
 
a
u
c
 
 
 
 
 
 
=
 
r
o
c
_
a
u
c
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
t
e
s
t
_
p
r
e
d
s
)

 
 
 
 
f
1
_
o
p
t
 
 
 
=
 
f
1
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
)

 
 
 
 
p
r
e
c
_
o
p
t
 
=
 
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
,
 
z
e
r
o
_
d
i
v
i
s
i
o
n
=
0
)

 
 
 
 
r
e
c
_
o
p
t
 
 
=
 
r
e
c
a
l
l
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
)

 
 
 
 
a
c
c
_
o
p
t
 
 
=
 
a
c
c
u
r
a
c
y
_
s
c
o
r
e
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
)


 
 
 
 
p
r
i
n
t
(
'
\
n
-
-
-
 
S
E
C
O
N
D
A
R
Y
:
 
B
i
n
a
r
y
 
c
o
n
n
e
c
t
i
v
i
t
y
 
t
a
s
k
 
-
-
-
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
P
R
-
A
U
C
 
 
 
 
 
 
 
 
:
 
{
p
r
_
a
u
c
:
.
4
f
}
 
 
 
(
p
r
i
m
a
r
y
 
f
o
r
 
i
m
b
a
l
a
n
c
e
d
 
d
a
t
a
)
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
R
O
C
-
A
U
C
 
 
 
 
 
 
 
:
 
{
a
u
c
:
.
4
f
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
F
1
 
@
 
t
h
r
=
{
o
p
t
_
t
h
r
:
.
3
f
}
 
:
 
{
f
1
_
o
p
t
:
.
4
f
}
'
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
P
r
e
c
i
s
i
o
n
 
 
 
 
 
:
 
{
p
r
e
c
_
o
p
t
:
.
4
f
}
 
 
|
 
 
R
e
c
a
l
l
:
 
{
r
e
c
_
o
p
t
:
.
4
f
}
 
 
|
 
 
A
c
c
:
 
{
a
c
c
_
o
p
t
:
.
4
f
}
'
)

 
 
 
 
p
r
i
n
t
(
)

 
 
 
 
p
r
i
n
t
(
c
l
a
s
s
i
f
i
c
a
t
i
o
n
_
r
e
p
o
r
t
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
a
r
g
e
t
_
n
a
m
e
s
=
[
'
N
o
 
C
o
v
e
r
a
g
e
'
,
 
'
H
a
s
 
C
o
v
e
r
a
g
e
'
]
)
)


 
 
 
 
#
 
─
─
 
F
i
g
u
r
e
 
2
:
 
P
r
e
d
i
c
t
e
d
 
v
s
 
a
c
t
u
a
l
 
+
 
R
e
s
i
d
u
a
l
 
h
i
s
t
o
g
r
a
m
 
─
─
─

 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
1
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
3
,
 
5
)
)

 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
R
e
g
r
e
s
s
i
o
n
 
Q
u
a
l
i
t
y
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
a
x
e
s
[
0
]
.
s
c
a
t
t
e
r
(
y
_
t
e
s
t
_
c
o
n
t
,
 
t
e
s
t
_
p
r
e
d
s
,
 
a
l
p
h
a
=
0
.
3
5
,
 
s
=
1
2
,
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
)

 
 
 
 
m
n
 
=
 
m
i
n
(
y
_
t
e
s
t
_
c
o
n
t
.
m
i
n
(
)
,
 
t
e
s
t
_
p
r
e
d
s
.
m
i
n
(
)
)

 
 
 
 
m
x
 
=
 
m
a
x
(
y
_
t
e
s
t
_
c
o
n
t
.
m
a
x
(
)
,
 
t
e
s
t
_
p
r
e
d
s
.
m
a
x
(
)
)

 
 
 
 
a
x
e
s
[
0
]
.
p
l
o
t
(
[
m
n
,
 
m
x
]
,
 
[
m
n
,
 
m
x
]
,
 
'
r
-
-
'
,
 
l
w
=
1
.
5
,
 
l
a
b
e
l
=
'
4
5
°
 
r
e
f
e
r
e
n
c
e
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
x
l
a
b
e
l
(
f
'
A
c
t
u
a
l
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
y
l
a
b
e
l
(
f
'
P
r
e
d
i
c
t
e
d
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
t
i
t
l
e
(

 
 
 
 
 
 
 
 
f
'
P
r
e
d
i
c
t
e
d
 
v
s
 
A
c
t
u
a
l
\
n
M
A
E
=
{
m
a
e
:
.
3
f
}
 
 
R
M
S
E
=
{
r
m
s
e
:
.
3
f
}
 
 
S
p
e
a
r
m
a
n
 
ρ
=
{
r
h
o
:
.
3
f
}
'
)

 
 
 
 
a
x
e
s
[
0
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
)


 
 
 
 
a
x
e
s
[
1
]
.
h
i
s
t
(
r
e
s
i
d
u
a
l
s
,
 
b
i
n
s
=
4
0
,
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
,
 
e
d
g
e
c
o
l
o
r
=
'
w
h
i
t
e
'
,
 
a
l
p
h
a
=
0
.
8
)

 
 
 
 
a
x
e
s
[
1
]
.
a
x
v
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
w
=
1
.
5
,
 
l
a
b
e
l
=
'
Z
e
r
o
'
)

 
 
 
 
a
x
e
s
[
1
]
.
a
x
v
l
i
n
e
(
r
e
s
i
d
u
a
l
s
.
m
e
a
n
(
)
,
 
c
o
l
o
r
=
'
o
r
a
n
g
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
'
,
 
l
w
=
1
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
M
e
a
n
 
=
 
{
r
e
s
i
d
u
a
l
s
.
m
e
a
n
(
)
:
.
3
f
}
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
R
e
s
i
d
u
a
l
 
(
p
r
e
d
i
c
t
e
d
 
−
 
t
r
u
e
)
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
C
o
u
n
t
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
t
i
t
l
e
(
f
'
R
e
s
i
d
u
a
l
 
D
i
s
t
r
i
b
u
t
i
o
n
 
 
(
s
t
d
 
=
 
{
r
e
s
i
d
u
a
l
s
.
s
t
d
(
)
:
.
3
f
}
)
'
)

 
 
 
 
a
x
e
s
[
1
]
.
l
e
g
e
n
d
(
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
a
v
e
f
i
g
(
f
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
r
e
g
r
e
s
s
i
o
n
.
p
n
g
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
#
 
─
─
 
F
i
g
u
r
e
 
3
:
 
R
e
s
i
d
u
a
l
s
 
v
s
 
t
r
u
e
 
v
a
l
u
e
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
f
i
g
,
 
a
x
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
f
i
g
s
i
z
e
=
(
9
,
 
5
)
)

 
 
 
 
a
x
.
s
c
a
t
t
e
r
(
y
_
t
e
s
t
_
c
o
n
t
,
 
r
e
s
i
d
u
a
l
s
,
 
a
l
p
h
a
=
0
.
3
5
,
 
s
=
1
2
,
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
)

 
 
 
 
a
x
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
w
=
1
.
5
,
 
l
a
b
e
l
=
'
Z
e
r
o
 
r
e
s
i
d
u
a
l
'
)


 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
b
i
n
_
m
e
a
n
s
,
 
b
i
n
_
e
d
g
e
s
,
 
_
 
=
 
b
i
n
n
e
d
_
s
t
a
t
i
s
t
i
c
(

 
 
 
 
 
 
 
 
 
 
 
 
y
_
t
e
s
t
_
c
o
n
t
,
 
r
e
s
i
d
u
a
l
s
,
 
s
t
a
t
i
s
t
i
c
=
'
m
e
a
n
'
,
 
b
i
n
s
=
1
0
)

 
 
 
 
 
 
 
 
b
i
n
_
c
e
n
t
e
r
s
 
=
 
(
b
i
n
_
e
d
g
e
s
[
:
-
1
]
 
+
 
b
i
n
_
e
d
g
e
s
[
1
:
]
)
 
/
 
2

 
 
 
 
 
 
 
 
a
x
.
p
l
o
t
(
b
i
n
_
c
e
n
t
e
r
s
,
 
b
i
n
_
m
e
a
n
s
,
 
'
o
-
'
,
 
c
o
l
o
r
=
'
o
r
a
n
g
e
'
,
 
l
w
=
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
m
s
=
6
,
 
l
a
b
e
l
=
'
B
i
n
 
m
e
a
n
 
r
e
s
i
d
u
a
l
'
)

 
 
 
 
e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
:

 
 
 
 
 
 
 
 
p
a
s
s


 
 
 
 
a
x
.
s
e
t
_
x
l
a
b
e
l
(
f
'
T
r
u
e
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
a
x
.
s
e
t
_
y
l
a
b
e
l
(
'
R
e
s
i
d
u
a
l
 
 
(
p
r
e
d
i
c
t
e
d
 
−
 
t
r
u
e
)
'
)

 
 
 
 
a
x
.
s
e
t
_
t
i
t
l
e
(

 
 
 
 
 
 
 
 
f
'
R
e
s
i
d
u
a
l
s
 
v
s
 
T
r
u
e
 
V
a
l
u
e
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
\
n
'

 
 
 
 
 
 
 
 
f
'
U
p
w
a
r
d
 
s
l
o
p
e
 
=
 
m
o
d
e
l
 
u
n
d
e
r
s
h
o
o
t
s
 
h
i
g
h
-
d
e
n
s
i
t
y
 
c
h
i
p
s
;
 
'

 
 
 
 
 
 
 
 
f
'
d
o
w
n
w
a
r
d
 
=
 
o
v
e
r
s
h
o
o
t
s
'
)

 
 
 
 
a
x
.
l
e
g
e
n
d
(
)
;
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
a
v
e
f
i
g
(
f
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
r
e
s
i
d
u
a
l
s
.
p
n
g
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
#
 
─
─
 
F
i
g
u
r
e
 
4
:
 
S
p
a
t
i
a
l
 
p
r
e
d
i
c
t
i
o
n
 
m
a
p
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
l
o
n
s
 
=
 
t
e
s
t
_
d
f
[
'
l
o
n
'
]
.
v
a
l
u
e
s

 
 
 
 
l
a
t
s
 
=
 
t
e
s
t
_
d
f
[
'
l
a
t
'
]
.
v
a
l
u
e
s

 
 
 
 
r
e
s
_
a
b
s
_
m
a
x
 
=
 
n
p
.
p
e
r
c
e
n
t
i
l
e
(
n
p
.
a
b
s
(
r
e
s
i
d
u
a
l
s
)
,
 
9
5
)


 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
1
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
6
,
 
6
)
)

 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
S
p
a
t
i
a
l
 
P
r
e
d
i
c
t
i
o
n
 
M
a
p
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
\
n
N
E
 
I
n
d
i
a
 
t
e
s
t
 
s
e
t
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
s
c
0
 
=
 
a
x
e
s
[
0
]
.
s
c
a
t
t
e
r
(
l
o
n
s
,
 
l
a
t
s
,
 
c
=
t
e
s
t
_
p
r
e
d
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
m
a
p
=
'
R
d
Y
l
G
n
'
,
 
s
=
1
8
,
 
a
l
p
h
a
=
0
.
7
5
)

 
 
 
 
p
l
t
.
c
o
l
o
r
b
a
r
(
s
c
0
,
 
a
x
=
a
x
e
s
[
0
]
,
 
l
a
b
e
l
=
f
'
P
r
e
d
i
c
t
e
d
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
L
o
n
g
i
t
u
d
e
'
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
L
a
t
i
t
u
d
e
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
t
i
t
l
e
(
'
P
r
e
d
i
c
t
e
d
 
v
a
l
u
e
'
)


 
 
 
 
s
c
1
 
=
 
a
x
e
s
[
1
]
.
s
c
a
t
t
e
r
(
l
o
n
s
,
 
l
a
t
s
,
 
c
=
r
e
s
i
d
u
a
l
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
m
a
p
=
'
R
d
B
u
_
r
'
,
 
s
=
1
8
,
 
a
l
p
h
a
=
0
.
7
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
v
m
i
n
=
-
r
e
s
_
a
b
s
_
m
a
x
,
 
v
m
a
x
=
r
e
s
_
a
b
s
_
m
a
x
)

 
 
 
 
p
l
t
.
c
o
l
o
r
b
a
r
(
s
c
1
,
 
a
x
=
a
x
e
s
[
1
]
,
 
l
a
b
e
l
=
'
R
e
s
i
d
u
a
l
 
(
p
r
e
d
i
c
t
e
d
 
−
 
t
r
u
e
)
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
L
o
n
g
i
t
u
d
e
'
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
L
a
t
i
t
u
d
e
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
t
i
t
l
e
(
'
R
e
s
i
d
u
a
l
s
 
 
(
b
l
u
e
 
=
 
u
n
d
e
r
-
p
r
e
d
i
c
t
e
d
,
 
r
e
d
 
=
 
o
v
e
r
-
p
r
e
d
i
c
t
e
d
)
'
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
a
v
e
f
i
g
(
f
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
s
p
a
t
i
a
l
.
p
n
g
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
#
 
─
─
 
F
i
g
u
r
e
 
5
:
 
B
i
n
a
r
y
 
c
o
n
n
e
c
t
i
v
i
t
y
 
t
a
s
k
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
1
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
2
,
 
5
)
)

 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
B
i
n
a
r
y
 
C
o
n
n
e
c
t
i
v
i
t
y
 
T
a
s
k
 
—
 
{
T
A
R
G
E
T
_
N
A
M
E
}
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
c
m
 
=
 
c
o
n
f
u
s
i
o
n
_
m
a
t
r
i
x
(
y
_
t
e
s
t
_
b
i
n
,
 
y
_
p
r
e
d
_
b
i
n
)

 
 
 
 
i
m
 
=
 
a
x
e
s
[
0
]
.
i
m
s
h
o
w
(
c
m
,
 
i
n
t
e
r
p
o
l
a
t
i
o
n
=
'
n
e
a
r
e
s
t
'
,
 
c
m
a
p
=
'
B
l
u
e
s
'
)

 
 
 
 
f
i
g
.
c
o
l
o
r
b
a
r
(
i
m
,
 
a
x
=
a
x
e
s
[
0
]
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
s
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
l
a
b
e
l
s
(
[
'
N
o
 
C
o
v
.
'
,
 
'
H
a
s
 
C
o
v
.
'
]
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
y
t
i
c
k
s
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
y
t
i
c
k
l
a
b
e
l
s
(
[
'
N
o
 
C
o
v
.
'
,
 
'
H
a
s
 
C
o
v
.
'
]
)

 
 
 
 
f
o
r
 
r
 
i
n
 
r
a
n
g
e
(
2
)
:

 
 
 
 
 
 
 
 
f
o
r
 
c
 
i
n
 
r
a
n
g
e
(
2
)
:

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
]
.
t
e
x
t
(
c
,
 
r
,
 
s
t
r
(
c
m
[
r
,
 
c
]
)
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
'
w
h
i
t
e
'
 
i
f
 
c
m
[
r
,
 
c
]
 
>
 
c
m
.
m
a
x
(
)
 
/
 
2
 
e
l
s
e
 
'
b
l
a
c
k
'
,
 
f
o
n
t
s
i
z
e
=
1
4
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
P
r
e
d
i
c
t
e
d
'
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
T
r
u
e
'
)

 
 
 
 
a
x
e
s
[
0
]
.
s
e
t
_
t
i
t
l
e
(
f
'
C
o
n
f
u
s
i
o
n
 
M
a
t
r
i
x
 
 
(
t
h
r
 
=
 
{
o
p
t
_
t
h
r
:
.
3
f
}
)
'
)


 
 
 
 
p
r
e
c
_
t
,
 
r
e
c
_
t
,
 
_
 
=
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
(
y
_
t
e
s
t
_
b
i
n
,
 
t
e
s
t
_
p
r
e
d
s
)

 
 
 
 
a
x
e
s
[
1
]
.
p
l
o
t
(
r
e
c
_
t
,
 
p
r
e
c
_
t
,
 
l
w
=
2
,
 
l
a
b
e
l
=
f
'
P
R
-
A
U
C
 
=
 
{
p
r
_
a
u
c
:
.
3
f
}
'
)

 
 
 
 
a
x
e
s
[
1
]
.
a
x
h
l
i
n
e
(
y
_
t
e
s
t
_
b
i
n
.
m
e
a
n
(
)
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
c
o
l
o
r
=
'
g
r
e
y
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
B
a
s
e
l
i
n
e
 
(
{
y
_
t
e
s
t
_
b
i
n
.
m
e
a
n
(
)
:
.
3
f
}
)
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
c
a
t
t
e
r
(
[
r
e
c
_
o
p
t
]
,
 
[
p
r
e
c
_
o
p
t
]
,
 
m
a
r
k
e
r
=
'
*
'
,
 
s
=
2
0
0
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
z
o
r
d
e
r
=
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
V
a
l
-
o
p
t
 
t
h
r
 
=
 
{
o
p
t
_
t
h
r
:
.
3
f
}
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
R
e
c
a
l
l
'
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
P
r
e
c
i
s
i
o
n
'
)

 
 
 
 
a
x
e
s
[
1
]
.
s
e
t
_
t
i
t
l
e
(
'
P
r
e
c
i
s
i
o
n
-
R
e
c
a
l
l
 
C
u
r
v
e
'
)

 
 
 
 
a
x
e
s
[
1
]
.
l
e
g
e
n
d
(
l
o
c
=
'
u
p
p
e
r
 
r
i
g
h
t
'
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
x
l
i
m
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
i
m
(
[
0
,
 
1
]
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
a
v
e
f
i
g
(
f
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
_
b
i
n
a
r
y
.
p
n
g
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
a
l
l
_
m
e
t
r
i
c
s
[
T
A
R
G
E
T
_
N
A
M
E
]
 
=
 
{

 
 
 
 
 
 
 
 
'
m
o
d
e
l
'
:
 
 
 
 
 
 
 
 
 
f
'
P
r
i
t
h
v
i
_
3
0
0
M
_
l
i
n
e
a
r
_
a
g
g
_
{
T
A
R
G
E
T
_
N
A
M
E
}
'
,

 
 
 
 
 
 
 
 
'
t
a
r
g
e
t
'
:
 
 
 
 
 
 
 
 
T
A
R
G
E
T
_
N
A
M
E
,

 
 
 
 
 
 
 
 
'
m
a
e
'
:
 
 
 
 
 
 
 
 
 
 
 
m
a
e
,

 
 
 
 
 
 
 
 
'
r
m
s
e
'
:
 
 
 
 
 
 
 
 
 
 
r
m
s
e
,

 
 
 
 
 
 
 
 
'
s
p
e
a
r
m
a
n
_
r
h
o
'
:
 
 
r
h
o
,

 
 
 
 
 
 
 
 
'
p
r
_
a
u
c
'
:
 
 
 
 
 
 
 
 
p
r
_
a
u
c
,

 
 
 
 
 
 
 
 
'
r
o
c
_
a
u
c
'
:
 
 
 
 
 
 
 
a
u
c
,

 
 
 
 
 
 
 
 
'
f
1
_
o
p
t
'
:
 
 
 
 
 
 
 
 
f
1
_
o
p
t
,

 
 
 
 
 
 
 
 
'
o
p
t
_
t
h
r
e
s
h
o
l
d
'
:
 
o
p
t
_
t
h
r
,

 
 
 
 
 
 
 
 
'
p
r
e
c
i
s
i
o
n
_
o
p
t
'
:
 
p
r
e
c
_
o
p
t
,

 
 
 
 
 
 
 
 
'
r
e
c
a
l
l
_
o
p
t
'
:
 
 
 
 
r
e
c
_
o
p
t
,

 
 
 
 
 
 
 
 
'
a
c
c
u
r
a
c
y
_
o
p
t
'
:
 
 
a
c
c
_
o
p
t
,

 
 
 
 
 
 
 
 
'
n
_
t
r
a
i
n
'
:
 
 
 
 
 
 
 
l
e
n
(
t
r
a
i
n
_
d
f
)
 
+
 
l
e
n
(
v
a
l
_
d
f
)
,

 
 
 
 
 
 
 
 
'
n
_
t
e
s
t
'
:
 
 
 
 
 
 
 
 
l
e
n
(
t
e
s
t
_
d
f
)
,

 
 
 
 
}


 
 
 
 
d
e
l
 
r
e
g
_
m
o
d
e
l

 
 
 
 
t
o
r
c
h
.
c
u
d
a
.
e
m
p
t
y
_
c
a
c
h
e
(
)

#
#
 
S
t
e
p
 
9
:
 
S
u
m
m
a
r
y
 
—
 
R
e
g
r
e
s
s
i
o
n
 
Q
u
a
l
i
t
y
 
+
 
B
i
n
a
r
y
 
T
a
s
k
 
C
o
m
p
a
r
i
s
o
n

In [ ]:
m
_
c
o
v
 
=
 
a
l
l
_
m
e
t
r
i
c
s
[
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
]

m
_
l
m
t
 
=
 
a
l
l
_
m
e
t
r
i
c
s
[
'
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
]


#
 
─
─
 
P
r
i
m
a
r
y
:
 
r
e
g
r
e
s
s
i
o
n
 
q
u
a
l
i
t
y
 
t
a
b
l
e
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

r
e
g
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
[

 
 
 
 
{
'
T
a
r
g
e
t
'
:
 
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
,
 
'
M
A
E
'
:
 
m
_
c
o
v
[
'
m
a
e
'
]
,
 
'
R
M
S
E
'
:
 
m
_
c
o
v
[
'
r
m
s
e
'
]
,

 
 
 
 
 
'
S
p
e
a
r
m
a
n
_
r
h
o
'
:
 
m
_
c
o
v
[
'
s
p
e
a
r
m
a
n
_
r
h
o
'
]
}
,

 
 
 
 
{
'
T
a
r
g
e
t
'
:
 
'
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
,
 
 
 
 
'
M
A
E
'
:
 
m
_
l
m
t
[
'
m
a
e
'
]
,
 
'
R
M
S
E
'
:
 
m
_
l
m
t
[
'
r
m
s
e
'
]
,

 
 
 
 
 
'
S
p
e
a
r
m
a
n
_
r
h
o
'
:
 
m
_
l
m
t
[
'
s
p
e
a
r
m
a
n
_
r
h
o
'
]
}
,

]
)

p
r
i
n
t
(
'
=
=
=
 
P
R
I
M
A
R
Y
:
 
R
e
g
r
e
s
s
i
o
n
 
q
u
a
l
i
t
y
 
=
=
=
'
)

p
r
i
n
t
(
r
e
g
_
d
f
.
t
o
_
s
t
r
i
n
g
(
i
n
d
e
x
=
F
a
l
s
e
)
)


#
 
─
─
 
S
e
c
o
n
d
a
r
y
:
 
b
i
n
a
r
y
 
t
a
s
k
 
c
o
m
p
a
r
i
s
o
n
 
w
i
t
h
 
N
B
0
7
 
b
a
s
e
l
i
n
e
 
─
─
─
─
─

n
b
0
7
_
p
a
t
h
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
r
e
s
u
l
t
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
f
t
_
m
e
t
r
i
c
s
.
c
s
v
'
)

i
f
 
n
b
0
7
_
p
a
t
h
.
e
x
i
s
t
s
(
)
:

 
 
 
 
b
a
s
e
 
 
 
 
 
 
=
 
p
d
.
r
e
a
d
_
c
s
v
(
n
b
0
7
_
p
a
t
h
)
.
i
l
o
c
[
0
]

 
 
 
 
b
a
s
e
_
p
r
 
 
 
=
 
b
a
s
e
.
g
e
t
(
'
p
r
_
a
u
c
'
,
 
 
 
f
l
o
a
t
(
'
n
a
n
'
)
)

 
 
 
 
b
a
s
e
_
a
u
c
 
 
=
 
b
a
s
e
.
g
e
t
(
'
r
o
c
_
a
u
c
'
,
 
 
f
l
o
a
t
(
'
n
a
n
'
)
)

 
 
 
 
b
a
s
e
_
f
1
 
 
 
=
 
b
a
s
e
.
g
e
t
(
'
f
1
_
o
p
t
'
,
 
 
 
f
l
o
a
t
(
'
n
a
n
'
)
)

e
l
s
e
:

 
 
 
 
b
a
s
e
_
p
r
 
=
 
b
a
s
e
_
a
u
c
 
=
 
b
a
s
e
_
f
1
 
=
 
f
l
o
a
t
(
'
n
a
n
'
)

 
 
 
 
p
r
i
n
t
(
'
N
B
0
7
 
b
a
s
e
l
i
n
e
 
n
o
t
 
f
o
u
n
d
 
—
 
r
u
n
 
N
B
0
7
 
f
i
r
s
t
 
f
o
r
 
f
u
l
l
 
c
o
m
p
a
r
i
s
o
n
.
'
)


b
i
n
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
[

 
 
 
 
{
'
M
o
d
e
l
'
:
 
'
N
B
0
7
 
b
i
n
a
r
y
 
l
a
b
e
l
'
,
 
 
 
 
 
 
 
 
 
'
P
R
-
A
U
C
'
:
 
b
a
s
e
_
p
r
,
 
 
 
 
 
 
 
 
 
'
R
O
C
-
A
U
C
'
:
 
b
a
s
e
_
a
u
c
,
 
 
 
 
 
 
 
 
 
'
F
1
'
:
 
b
a
s
e
_
f
1
}
,

 
 
 
 
{
'
M
o
d
e
l
'
:
 
'
N
B
1
4
 
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
,
 
 
 
 
'
P
R
-
A
U
C
'
:
 
m
_
c
o
v
[
'
p
r
_
a
u
c
'
]
,
 
'
R
O
C
-
A
U
C
'
:
 
m
_
c
o
v
[
'
r
o
c
_
a
u
c
'
]
,
 
'
F
1
'
:
 
m
_
c
o
v
[
'
f
1
_
o
p
t
'
]
}
,

 
 
 
 
{
'
M
o
d
e
l
'
:
 
'
N
B
1
4
 
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
,
 
 
 
 
 
 
 
'
P
R
-
A
U
C
'
:
 
m
_
l
m
t
[
'
p
r
_
a
u
c
'
]
,
 
'
R
O
C
-
A
U
C
'
:
 
m
_
l
m
t
[
'
r
o
c
_
a
u
c
'
]
,
 
'
F
1
'
:
 
m
_
l
m
t
[
'
f
1
_
o
p
t
'
]
}
,

]
)

p
r
i
n
t
(
'
\
n
=
=
=
 
S
E
C
O
N
D
A
R
Y
:
 
B
i
n
a
r
y
 
c
o
n
n
e
c
t
i
v
i
t
y
 
t
a
s
k
 
=
=
=
'
)

p
r
i
n
t
(
b
i
n
_
d
f
.
t
o
_
s
t
r
i
n
g
(
i
n
d
e
x
=
F
a
l
s
e
)
)


#
 
─
─
 
F
i
g
u
r
e
:
 
t
w
o
-
p
a
n
e
l
 
s
u
m
m
a
r
y
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
1
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
4
,
 
5
)
)

f
i
g
.
s
u
p
t
i
t
l
e
(
'
P
r
i
t
h
v
i
-
3
0
0
M
 
L
i
n
e
a
r
 
—
 
A
g
g
r
e
g
a
t
e
-
L
a
b
e
l
 
B
e
n
c
h
m
a
r
k
\
n
N
E
 
I
n
d
i
a
 
t
e
s
t
 
s
e
t
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


#
 
L
e
f
t
 
p
a
n
e
l
:
 
r
e
g
r
e
s
s
i
o
n
 
m
e
t
r
i
c
s
 
(
M
A
E
 
+
 
R
M
S
E
 
b
a
r
s
,
 
S
p
e
a
r
m
a
n
 
ρ
 
a
n
n
o
t
a
t
e
d
)

t
a
r
g
e
t
s
 
 
 
=
 
[
'
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
'
,
 
'
l
o
g
_
m
e
a
n
_
t
e
s
t
s
'
]

m
a
e
_
v
a
l
s
 
 
=
 
[
m
_
c
o
v
[
'
m
a
e
'
]
,
 
 
m
_
l
m
t
[
'
m
a
e
'
]
]

r
m
s
e
_
v
a
l
s
 
=
 
[
m
_
c
o
v
[
'
r
m
s
e
'
]
,
 
m
_
l
m
t
[
'
r
m
s
e
'
]
]

r
h
o
_
v
a
l
s
 
 
=
 
[
m
_
c
o
v
[
'
s
p
e
a
r
m
a
n
_
r
h
o
'
]
,
 
m
_
l
m
t
[
'
s
p
e
a
r
m
a
n
_
r
h
o
'
]
]

x
,
 
w
 
=
 
n
p
.
a
r
a
n
g
e
(
l
e
n
(
t
a
r
g
e
t
s
)
)
,
 
0
.
3
5

a
x
e
s
[
0
]
.
b
a
r
(
x
 
-
 
w
/
2
,
 
m
a
e
_
v
a
l
s
,
 
 
w
,
 
l
a
b
e
l
=
'
M
A
E
'
,
 
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
)

a
x
e
s
[
0
]
.
b
a
r
(
x
 
+
 
w
/
2
,
 
r
m
s
e
_
v
a
l
s
,
 
w
,
 
l
a
b
e
l
=
'
R
M
S
E
'
,
 
c
o
l
o
r
=
'
#
D
D
8
4
5
2
'
)

f
o
r
 
i
,
 
(
m
v
,
 
r
v
,
 
r
h
o
_
v
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
z
i
p
(
m
a
e
_
v
a
l
s
,
 
r
m
s
e
_
v
a
l
s
,
 
r
h
o
_
v
a
l
s
)
)
:

 
 
 
 
a
x
e
s
[
0
]
.
t
e
x
t
(
i
 
-
 
w
/
2
,
 
m
v
 
 
+
 
m
a
x
(
m
a
e
_
v
a
l
s
 
+
 
r
m
s
e
_
v
a
l
s
)
 
*
 
0
.
0
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
'
{
m
v
:
.
3
f
}
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
b
o
t
t
o
m
'
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
e
s
[
0
]
.
t
e
x
t
(
i
 
+
 
w
/
2
,
 
r
v
 
 
+
 
m
a
x
(
m
a
e
_
v
a
l
s
 
+
 
r
m
s
e
_
v
a
l
s
)
 
*
 
0
.
0
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
'
{
r
v
:
.
3
f
}
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
b
o
t
t
o
m
'
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
e
s
[
0
]
.
t
e
x
t
(
i
,
 
m
a
x
(
m
v
,
 
r
v
)
 
*
 
1
.
1
8
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
'
ρ
 
=
 
{
r
h
o
_
v
:
.
3
f
}
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
b
o
t
t
o
m
'
,
 
f
o
n
t
s
i
z
e
=
9
,
 
c
o
l
o
r
=
'
#
5
5
A
8
6
8
'
)

a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
s
(
x
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
l
a
b
e
l
s
(
t
a
r
g
e
t
s
,
 
r
o
t
a
t
i
o
n
=
1
0
,
 
h
a
=
'
r
i
g
h
t
'
)

a
x
e
s
[
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
E
r
r
o
r
 
(
l
o
w
e
r
 
i
s
 
b
e
t
t
e
r
)
'
)

a
x
e
s
[
0
]
.
s
e
t
_
t
i
t
l
e
(
'
P
r
i
m
a
r
y
:
 
R
e
g
r
e
s
s
i
o
n
 
Q
u
a
l
i
t
y
'
)

a
x
e
s
[
0
]
.
l
e
g
e
n
d
(
)

a
x
e
s
[
0
]
.
s
e
t
_
y
l
i
m
(
0
,
 
m
a
x
(
m
a
e
_
v
a
l
s
 
+
 
r
m
s
e
_
v
a
l
s
)
 
*
 
1
.
4
5
)


#
 
R
i
g
h
t
 
p
a
n
e
l
:
 
b
i
n
a
r
y
 
t
a
s
k
 
m
e
t
r
i
c
s
 
v
s
 
N
B
0
7
 
b
a
s
e
l
i
n
e

m
o
d
e
l
s
 
=
 
b
i
n
_
d
f
[
'
M
o
d
e
l
'
]
.
t
o
l
i
s
t
(
)

x
2
,
 
w
2
 
=
 
n
p
.
a
r
a
n
g
e
(
l
e
n
(
m
o
d
e
l
s
)
)
,
 
0
.
2
5

a
x
e
s
[
1
]
.
b
a
r
(
x
2
 
-
 
w
2
,
 
b
i
n
_
d
f
[
'
P
R
-
A
U
C
'
]
.
f
i
l
l
n
a
(
0
)
,
 
 
w
2
,
 
l
a
b
e
l
=
'
P
R
-
A
U
C
'
,
 
 
c
o
l
o
r
=
'
#
4
C
7
2
B
0
'
)

a
x
e
s
[
1
]
.
b
a
r
(
x
2
,
 
 
 
 
 
 
b
i
n
_
d
f
[
'
R
O
C
-
A
U
C
'
]
.
f
i
l
l
n
a
(
0
)
,
 
w
2
,
 
l
a
b
e
l
=
'
R
O
C
-
A
U
C
'
,
 
c
o
l
o
r
=
'
#
D
D
8
4
5
2
'
)

a
x
e
s
[
1
]
.
b
a
r
(
x
2
 
+
 
w
2
,
 
b
i
n
_
d
f
[
'
F
1
'
]
.
f
i
l
l
n
a
(
0
)
,
 
 
 
 
 
 
w
2
,
 
l
a
b
e
l
=
'
F
1
'
,
 
 
 
 
 
 
c
o
l
o
r
=
'
#
5
5
A
8
6
8
'
)

a
x
e
s
[
1
]
.
s
e
t
_
x
t
i
c
k
s
(
x
2
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
x
t
i
c
k
l
a
b
e
l
s
(
m
o
d
e
l
s
,
 
r
o
t
a
t
i
o
n
=
1
2
,
 
h
a
=
'
r
i
g
h
t
'
)

a
x
e
s
[
1
]
.
s
e
t
_
y
l
i
m
(
0
,
 
1
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
S
c
o
r
e
'
)

a
x
e
s
[
1
]
.
s
e
t
_
t
i
t
l
e
(
'
S
e
c
o
n
d
a
r
y
:
 
B
i
n
a
r
y
 
C
o
n
n
e
c
t
i
v
i
t
y
 
T
a
s
k
'
)

a
x
e
s
[
1
]
.
l
e
g
e
n
d
(
)


p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

p
l
t
.
s
a
v
e
f
i
g
(
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
m
p
a
r
i
s
o
n
.
p
n
g
'
,
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

p
l
t
.
s
h
o
w
(
)

## Step 10: Save Metrics & Push to GitHub

In [ ]:
for TARGET_NAME, m in all_metrics.items():
    pd.DataFrame([m]).to_csv(
        f'{RESULTS_DIR}/prithvi_linear_agg_{TARGET_NAME}_metrics.csv', index=False)
    # Save checkpoint path pointer
    if TARGET_NAME in best_ckpt_paths:
        with open(f'outputs/models/prithvi_linear_agg_{TARGET_NAME}_ckpt.txt', 'w') as f:
            f.write(best_ckpt_paths[TARGET_NAME])

print('Metrics saved.')

In [ ]:
i
m
p
o
r
t
 
s
u
b
p
r
o
c
e
s
s
,
 
o
s


t
o
k
e
n
 
=
 
"
Y
O
U
R
_
T
O
K
E
N
_
H
E
R
E
"

r
e
p
o
 
 
=
 
"
t
a
t
y
a
n
a
2
1
1
1
1
/
i
n
t
e
r
n
e
t
-
c
o
n
n
e
c
t
i
v
i
t
y
-
p
r
e
d
i
c
t
i
o
n
"


s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
c
o
n
f
i
g
'
,
 
'
-
-
g
l
o
b
a
l
'
,
 
'
u
s
e
r
.
e
m
a
i
l
'
,
 
'
t
a
t
y
a
n
a
2
1
1
z
y
@
o
u
t
l
o
o
k
.
c
o
m
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)

s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
c
o
n
f
i
g
'
,
 
'
-
-
g
l
o
b
a
l
'
,
 
'
u
s
e
r
.
n
a
m
e
'
,
 
 
'
t
a
t
y
a
n
a
2
1
1
1
1
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)

s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
r
e
m
o
t
e
'
,
 
'
s
e
t
-
u
r
l
'
,
 
'
o
r
i
g
i
n
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
'
h
t
t
p
s
:
/
/
{
t
o
k
e
n
}
@
g
i
t
h
u
b
.
c
o
m
/
{
r
e
p
o
}
.
g
i
t
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)


f
i
l
e
s
 
=
 
[

 
 
 
 
'
o
u
t
p
u
t
s
/
r
e
s
u
l
t
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
m
e
t
r
i
c
s
.
c
s
v
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
r
e
s
u
l
t
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
m
e
t
r
i
c
s
.
c
s
v
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
v
a
l
_
l
o
s
s
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
r
e
g
r
e
s
s
i
o
n
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
r
e
s
i
d
u
a
l
s
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
s
p
a
t
i
a
l
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
v
e
r
a
g
e
_
f
r
a
c
t
i
o
n
_
b
i
n
a
r
y
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
v
a
l
_
l
o
s
s
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
r
e
g
r
e
s
s
i
o
n
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
r
e
s
i
d
u
a
l
s
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
s
p
a
t
i
a
l
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
o
g
_
m
e
a
n
_
t
e
s
t
s
_
b
i
n
a
r
y
.
p
n
g
'
,

 
 
 
 
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
/
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
c
o
m
p
a
r
i
s
o
n
.
p
n
g
'
,

 
 
 
 
'
n
o
t
e
b
o
o
k
s
/
1
4
_
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
a
g
g
_
l
a
b
e
l
.
i
p
y
n
b
'
,

]

f
o
r
 
f
 
i
n
 
f
i
l
e
s
:

 
 
 
 
i
f
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
f
)
:

 
 
 
 
 
 
 
 
s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
a
d
d
'
,
 
'
-
f
'
,
 
f
]
,
 
c
h
e
c
k
=
T
r
u
e
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
'
S
k
i
p
p
i
n
g
 
(
n
o
t
 
y
e
t
 
g
e
n
e
r
a
t
e
d
)
:
 
{
f
}
'
)


d
i
f
f
 
=
 
s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
d
i
f
f
'
,
 
'
-
-
c
a
c
h
e
d
'
,
 
'
-
-
q
u
i
e
t
'
]
,
 
c
a
p
t
u
r
e
_
o
u
t
p
u
t
=
T
r
u
e
)

i
f
 
d
i
f
f
.
r
e
t
u
r
n
c
o
d
e
 
!
=
 
0
:

 
 
 
 
s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
c
o
m
m
i
t
'
,
 
'
-
m
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
'
P
h
a
s
e
 
1
B
:
 
P
r
i
t
h
v
i
-
3
0
0
M
 
l
i
n
e
a
r
 
a
g
g
r
e
g
a
t
e
-
l
a
b
e
l
 
b
e
n
c
h
m
a
r
k
 
(
N
B
1
4
)
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)

e
l
s
e
:

 
 
 
 
p
r
i
n
t
(
'
N
o
t
h
i
n
g
 
s
t
a
g
e
d
 
—
 
a
l
l
 
o
u
t
p
u
t
s
 
m
a
y
 
a
l
r
e
a
d
y
 
b
e
 
c
o
m
m
i
t
t
e
d
.
'
)


s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
p
u
l
l
'
,
 
'
-
-
r
e
b
a
s
e
'
,
 
'
o
r
i
g
i
n
'
,
 
'
m
a
i
n
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)

s
u
b
p
r
o
c
e
s
s
.
r
u
n
(
[
'
g
i
t
'
,
 
'
p
u
s
h
'
]
,
 
c
h
e
c
k
=
T
r
u
e
)

p
r
i
n
t
(
'
P
u
s
h
 
s
u
c
c
e
s
s
f
u
l
.
'
)